# VadCLIP Baseline — Runner (Kaggle)

Bản Kaggle của `run_baseline_colab.ipynb`. Từ §4 trở đi mọi thứ giống hệt bản Colab —
chỉ §1 và §3 khác, vì Kaggle không có Drive.

## Chuẩn bị: tạo 2 Dataset trước khi mở notebook

**Dataset 1 — `vadclip-baseline`** (~1.4 GB), cấu trúc đúng như sau:

```
baseline/                    ← copy từ VadCLIP/baseline/
list/                        ← copy từ VadCLIP/list/ (PHẢI có gt_ucf.npy,
                                gt_segment_ucf.npy, gt_label_ucf.npy)
model_ucf.pth                ← checkpoint tác giả công bố
model_baseline_ctrl.pth      ← baseline cũ của bạn (lambda = 0)
model_v0.pth                 ← thí nghiệm cũ của bạn (lambda = 0.01)
ViT-B-16.pt                  ← tuỳ chọn, xem bên dưới
```

**Dataset 2 — `ucf-clip-features`**: thư mục feature, tức là các thư mục lớp
`Abuse/`, `Arson/`, ..., `Normal_Videos_event/` chứa file `.npy`.

Notebook **tự dò** hai dataset này trong `/kaggle/input`, nên tên slug đặt gì cũng được.

## Cài đặt Session

| Mục | Giá trị | Vì sao |
|---|---|---|
| Accelerator | **GPU** (T4 hoặc P100) | `layers.py` upstream hardcode `.to('cuda')` |
| Internet | **On** | để `pip install ftfy` chạy được |
| Persistence | Files + Variables | giữ `/kaggle/working` giữa các session |

### `ViT-B-16.pt` — tránh phụ thuộc mạng

`model.py` gọi `clip.load("ViT-B/16")`, vốn tải 335 MB từ `openaipublic.azureedge.net` mỗi
session. §3 sẽ nạp sẵn file đó vào `~/.cache/clip/` nếu bạn bỏ nó vào Dataset 1
(`clip.load` kiểm tra SHA256 rồi bỏ qua bước tải). Không có cũng chạy được, miễn Internet On.

Tải một lần bằng: `wget https://openaipublic.azureedge.net/clip/models/5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f/ViT-B-16.pt`

## ⚠️ Giới hạn thời gian của Kaggle

Session GPU tối đa **12 giờ**, quota **30 giờ/tuần**. Lần train 10 epoch phải nằm gọn trong 12 giờ.

`--use-checkpoint` của upstream **không resume được** (nó nạp lại `epoch` nhưng vòng lặp vẫn là
`for e in range(max_epoch)`, tức chạy lại từ đầu). Nên nếu quá 12 giờ là mất trắng lần chạy.

Cách an toàn: bấm **Save Version → Save & Run All (Commit)** để chạy ở chế độ batch. Nó chạy nền,
không cần giữ trình duyệt, và output tự lưu.

## 1. Cấu Hình

Tự dò dataset trong `/kaggle/input`. Nếu dò sai thì điền tay vào ba biến `*_OVERRIDE`.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK       = Path('/kaggle/working')
TEMP       = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else Path('/tmp')

# Điền tay nếu tự dò sai. Ví dụ: Path('/kaggle/input/vadclip-baseline')
CODE_OVERRIDE    = None
FEATURE_OVERRIDE = None
CKPT_OVERRIDE    = None


def find_in_input(*markers, maxdepth=3):
    '''Dò thư mục con của /kaggle/input chứa đủ các đường dẫn đánh dấu.'''
    if not INPUT_ROOT.exists():
        return None
    queue = [(INPUT_ROOT, 0)]
    while queue:
        directory, depth = queue.pop(0)
        if all((directory / m).exists() for m in markers):
            return directory
        if depth < maxdepth:
            try:
                queue.extend((p, depth + 1) for p in sorted(directory.iterdir()) if p.is_dir())
            except (PermissionError, OSError):
                pass
    return None


CODE_ROOT = CODE_OVERRIDE or find_in_input('baseline/src/ucf_train.py', 'list/gt_ucf.npy')
if CODE_ROOT is None:
    CODE_ROOT = find_in_input('baseline/src/ucf_train.py')          # thiếu gt -> preflight sẽ báo
FEATURE_ROOT_INPUT = FEATURE_OVERRIDE or find_in_input('Abuse', 'Vandalism')
CKPT_ROOT = CKPT_OVERRIDE or find_in_input('model_ucf.pth') or CODE_ROOT

if CODE_ROOT is None:
    print('Có gì trong /kaggle/input:')
    for p in sorted(INPUT_ROOT.glob('*/*'))[:40]:
        print('  ', p)
    raise FileNotFoundError('Không tìm thấy dataset code. Điền tay vào CODE_OVERRIDE.')

# --- Code phải nằm ở nơi ghi được: /kaggle/input là read-only ---
BASELINE_SRC = WORK / 'baseline' / 'src'
LIST_DIR     = CODE_ROOT / 'list'
if not BASELINE_SRC.exists():
    print('Copy code sang thư mục ghi được ...')
    shutil.copytree(CODE_ROOT / 'baseline', WORK / 'baseline')
print('Code       :', BASELINE_SRC)

FEATURE_ROOT     = FEATURE_ROOT_INPUT     # đọc thẳng, /kaggle/input đã là đĩa local
PAPER_CHECKPOINT = CKPT_ROOT / 'model_ucf.pth' if CKPT_ROOT else None
OLD_MODEL_DIR    = CKPT_ROOT

RESULT_DIR = WORK / 'results'
LOG_DIR    = RESULT_DIR / 'logs'
OUT_ROOT   = WORK / 'baseline_out'

TRAIN_LIST = str(LIST_DIR / 'ucf_CLIP_rgb_relative.csv')
TEST_LIST  = str(LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv')
GT_ARGS = [
    '--gt-path',         str(LIST_DIR / 'gt_ucf.npy'),
    '--gt-segment-path', str(LIST_DIR / 'gt_segment_ucf.npy'),
    '--gt-label-path',   str(LIST_DIR / 'gt_label_ucf.npy'),
]

# ================= CẤU HÌNH BASELINE CHUẨN =================
# Giống hệt bản Colab. Đổi một giá trị ở đây là đổi cho mọi lần chạy.
SEEDS           = [777]
BASELINE_LR     = '2e-5'
BASELINE_EPOCHS = 10
BASELINE_BATCH  = 64
BASELINE_FLAGS  = ['--num-workers', '4', '--pin-memory', 'true', '--deterministic', 'true']
SAVE_EPOCH_CHECKPOINTS = False    # mỗi file ~350 MB; /kaggle/working giới hạn 20 GB
# ===========================================================

PAPER_METRICS = {'AUC1': 88.02, 'AP1': 33.56, 'AUC2': 85.69, 'AP2': 26.50, 'avgMAP': 6.68}

os.chdir(BASELINE_SRC)
sys.path.insert(0, str(BASELINE_SRC))
LOG_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

PY = [sys.executable, '-u']


def run_command(cmd, log_name=None):
    '''Chạy lệnh, stream từng dòng, ghi log ra /kaggle/working ngay khi có dòng mới.'''
    cmd = [str(part) for part in cmd]
    print('$', ' '.join(cmd), flush=True)
    log_file = open(LOG_DIR / log_name, 'w', encoding='utf-8') if log_name else None
    if log_file:
        log_file.write('$ ' + ' '.join(cmd) + chr(10))
        log_file.flush()
    captured = []
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
            captured.append(line)
            if log_file:
                log_file.write(line)
                log_file.flush()
    finally:
        process.wait()
        if log_file:
            log_file.close()
    if log_name:
        print('Log:', LOG_DIR / log_name)
    if process.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {process.returncode}')
    return ''.join(captured)


def score_model(model_path, name):
    '''Chấm điểm một file trọng số bằng ucf_test.py — thước đo duy nhất của notebook này.'''
    preflight()
    return run_command(PY + [
        'ucf_test.py',
        '--feature-root', FEATURE_ROOT,
        '--test-list', TEST_LIST,
        *GT_ARGS,
        '--model-path', model_path,
    ], log_name=f'score_{name}.log')


def train_baseline(seed):
    '''Train một baseline với cấu hình chuẩn ở trên. Trả về tag.'''
    preflight()
    tag = f'baseline_seed{seed}'
    out = OUT_ROOT / tag
    out.mkdir(parents=True, exist_ok=True)
    cmd = PY + [
        'ucf_train.py',
        '--feature-root', FEATURE_ROOT,
        '--train-list', TRAIN_LIST,
        '--test-list',  TEST_LIST,
        *GT_ARGS,
        '--seed', seed,
        '--lr', BASELINE_LR,
        '--max-epoch', BASELINE_EPOCHS,
        '--batch-size', BASELINE_BATCH,
        '--model-path',      out / f'model_{tag}.pth',
        '--checkpoint-path', out / 'checkpoint.pth',
        '--save-cur-path',   out / 'model_cur.pth',
    ] + BASELINE_FLAGS
    if SAVE_EPOCH_CHECKPOINTS:
        cmd += ['--epoch-checkpoint-dir', str(out / 'epoch_checkpoints')]
    run_command(cmd, log_name=f'train_{tag}.log')
    return tag


print('Feature    :', FEATURE_ROOT)
print('List       :', LIST_DIR)
print('Checkpoints:', OLD_MODEL_DIR)
print('Kết quả về :', RESULT_DIR)
print('cwd        :', Path.cwd())
print()
print('Cấu hình chuẩn: seeds', SEEDS, '| lr', BASELINE_LR, '| epoch', BASELINE_EPOCHS,
      '| batch', BASELINE_BATCH)
print('               ', ' '.join(BASELINE_FLAGS))

## 2. Cài Dependencies

Ảnh Kaggle đã có sẵn torch, sklearn, scipy, pandas, matplotlib. Chỉ thiếu `ftfy` và `regex`
mà tokenizer của CLIP cần. Cần bật **Internet: On**.

In [ ]:
!pip -q install ftfy regex

## 3. Nạp Sẵn Trọng Số CLIP

`model.py` gọi `clip.load("ViT-B/16")`, vốn tải 335 MB mỗi session. `clip._download` kiểm tra
`~/.cache/clip/ViT-B-16.pt` bằng SHA256 trước — nên copy file vào đó là bỏ qua được bước tải,
mà **không phải sửa `model.py`** (file này giữ nguyên verbatim upstream).

Không có `ViT-B-16.pt` trong dataset thì cell này bỏ qua, và CLIP sẽ tự tải khi train.

In [ ]:
CLIP_SHA256 = '5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f'
cache_dir = Path.home() / '.cache' / 'clip'
cache_dir.mkdir(parents=True, exist_ok=True)
target = cache_dir / 'ViT-B-16.pt'

source = next(INPUT_ROOT.glob('**/ViT-B-16.pt'), None)
if target.exists():
    print('Đã có sẵn trong cache:', target)
elif source is None:
    print('Không thấy ViT-B-16.pt trong /kaggle/input.')
    print('CLIP sẽ tự tải khi train (cần Internet: On).')
else:
    print('Copy', source, '->', target)
    shutil.copy2(source, target)

if target.exists():
    import hashlib
    digest = hashlib.sha256(target.read_bytes()).hexdigest()
    print('SHA256 khớp:', digest == CLIP_SHA256)
    if digest != CLIP_SHA256:
        print('  KHÔNG khớp -> clip.load sẽ tải lại. Kiểm tra lại file trong dataset.')

## 4. Preflight

Kiểm tra mọi thứ trước khi tiêu tốn hàng giờ GPU. `preflight()` được gọi tự động ở đầu mọi cell
train và chấm điểm, nên không thể chạy nhầm với setup hỏng.

Nếu báo **thiếu cờ `--deterministic`** → dataset đang giữ bản `baseline/` cũ. Cập nhật lại
Dataset 1 (New Version), rồi xoá `/kaggle/working/baseline` và chạy lại §1.

In [ ]:
import csv
import numpy as np
import torch
from collections import Counter

_preflight_done = False


def preflight(force=False):
    global _preflight_done
    if _preflight_done and not force:
        return True

    problems = []

    if not torch.cuda.is_available():
        problems.append('Không có GPU. Settings -> Accelerator -> GPU T4 x2 hoặc P100.')

    if FEATURE_ROOT is None:
        problems.append('Không tìm thấy dataset feature. Điền tay vào FEATURE_OVERRIDE ở §1.')

    need = [BASELINE_SRC / n for n in
            ['ucf_train.py', 'ucf_test.py', 'ucf_option.py', 'model.py',
             'utils/dataset.py', 'utils/tools.py', 'utils/layers.py',
             'utils/ucf_detectionMAP.py', 'clip/clip.py', 'clip/bpe_simple_vocab_16e6.txt.gz']]
    need += [LIST_DIR / 'ucf_CLIP_rgb_relative.csv', LIST_DIR / 'ucf_CLIP_rgbtest_relative.csv']
    for p in need:
        if not p.exists():
            problems.append(f'Thiếu file: {p}')

    for p in [LIST_DIR / n for n in ['gt_ucf.npy', 'gt_segment_ucf.npy', 'gt_label_ucf.npy']]:
        if not p.exists():
            problems.append(f'Thiếu ground truth: {p}')

    if (BASELINE_SRC / 'ucf_train.py').exists():
        helptext = subprocess.run(PY + ['ucf_train.py', '--help'],
                                  capture_output=True, text=True).stdout
        for flag in ['--num-workers', '--pin-memory', '--deterministic', '--feature-root']:
            if flag not in helptext:
                problems.append(f'ucf_train.py thiếu cờ {flag} — bản cũ. Cập nhật Dataset 1, '
                                'xoá /kaggle/working/baseline, chạy lại §1.')

    if FEATURE_ROOT is not None:
        for name, expected in [('ucf_CLIP_rgb_relative.csv', 16100),
                               ('ucf_CLIP_rgbtest_relative.csv', 290)]:
            path = LIST_DIR / name
            if not path.exists():
                continue
            rows = list(csv.DictReader(open(path, encoding='utf-8')))
            missing = [r for r in rows if not (FEATURE_ROOT / r['path']).exists()]
            print(f'  {name}: {len(rows)} dòng (mong đợi {expected}), thiếu {len(missing)}')
            if missing:
                print('    Thiếu theo nhãn:', dict(Counter(r['label'] for r in missing)))
                for r in missing[:5]:
                    print('      ', r['path'])
                problems.append(f'{name}: thiếu {len(missing)} file feature.')

    if problems:
        print()
        print('PREFLIGHT KHÔNG ĐẠT:')
        for p in problems:
            print('  -', p)
        raise RuntimeError('Sửa các mục trên rồi chạy lại cell này.')

    print()
    print('PREFLIGHT ĐẠT')
    print('  GPU          :', torch.cuda.get_device_name(0))
    print('  torch        :', torch.__version__)
    print('  FEATURE_ROOT :', FEATURE_ROOT)
    gt = np.load(LIST_DIR / 'gt_ucf.npy')
    print('  gt_ucf.npy   :', len(gt), 'frame |', int(gt.sum()), 'frame bất thường')
    free = shutil.disk_usage(WORK).free / 1e9
    print(f'  /kaggle/working còn trống: {free:.1f} GB')
    _preflight_done = True
    return True


preflight(force=True)

## 5. Chấm Các Model Đã Có

**Rẻ và quan trọng.** Vài phút, và nó sửa được bảng kết quả trong báo cáo.

| Model | Là gì | Mong đợi |
|---|---|---|
| `model_ucf.pth` | trọng số **tác giả công bố** | phải ra đúng **88.02 / 6.68** |
| `model_baseline_ctrl.pth` | bạn train, ràng buộc **TẮT** (λ=0) → baseline | ~88.13 / 8.50 |
| `model_v0.pth` | bạn train, ràng buộc **BẬT** (λ=0.01) → thí nghiệm | chưa biết |

Model đầu tiên là **quả cân chuẩn**. Ra đúng 88.02 nghĩa là thước đo, ground truth, test list và
bộ prompt đều đúng — và mọi con số còn lại đáng tin.

> Nhớ: `baseline_ctrl` là λ=**0**, `v0` là λ=**0.01**. Đừng đảo hai cái này.

In [ ]:
TARGETS = [
    (PAPER_CHECKPOINT,                          'paper'),
    (OLD_MODEL_DIR / 'model_baseline_ctrl.pth', 'baseline_ctrl'),
    (OLD_MODEL_DIR / 'model_v0.pth',            'v0'),
]

for path, name in TARGETS:
    print('#' * 78)
    print(name, '->', path)
    if path is None or not Path(path).exists():
        print('  BỎ QUA: không tìm thấy file.')
        continue
    score_model(path, name)

In [ ]:
# Kiểm tra thước đo: bản chấm điểm có tái lập đúng số của paper không?
import re


def read_score(name):
    path = LOG_DIR / f'score_{name}.log'
    if not path.exists():
        return None
    text = path.read_text(encoding='utf-8')
    def last(p, scale=1.0):
        hits = re.findall(p, text)
        return float(hits[-1]) * scale if hits else float('nan')
    return {'AUC1': last(r'AUC1:\s+([\d.]+)', 100), 'AP1': last(r'AP1:\s+([\d.]+)', 100),
            'AUC2': last(r'AUC2:\s+([\d.]+)', 100), 'AP2': last(r'AP2:\s*([\d.]+)', 100),
            'avgMAP': last(r'average MAP:\s+([\d.]+)')}


paper = read_score('paper')
if paper is None:
    print('Chưa chấm checkpoint tác giả — chạy cell trên trước.')
else:
    print('KIỂM TRA THƯỚC ĐO trên checkpoint tác giả:')
    ok = True
    for key, expected in PAPER_METRICS.items():
        got = paper[key]
        good = abs(got - expected) < 0.05
        ok = ok and good
        print(f'  {"OK  " if good else "LỆCH"} {key:7s} đo được {got:6.2f}   paper {expected:6.2f}')
    print()
    print('=> Thước đo chuẩn. Mọi con số khác trong notebook này đáng tin.' if ok else
          '=> Thước đo LỆCH. Dừng lại: kiểm tra ground truth và test list trước khi đọc số nào khác.')

## 6. Train Baseline Seed Mới

Cấu hình lấy từ §1, không nhận tham số riêng ở đây — để mọi lần chạy giống hệt nhau trừ seed.

```
lr 2e-5 · batch 64 · 10 epoch · MultiStepLR([4,8], 0.1)
--num-workers 4 --pin-memory true --deterministic true
```

- **lr 2e-5** là mặc định repo. Hai lần chạy trước cho 87.55 và 88.13, ôm trọn 88.02 của paper.
  Paper ghi 1e-5 nhưng chưa ai thử — để làm thí nghiệm riêng, đừng trộn vào baseline.
- **num-workers 4** khớp với `baseline_ctrl` và `v0`, nên mọi thứ so được với nhau. Chọn một
  giá trị rồi **không đổi nữa** — nó làm đổi thứ tự lô dữ liệu, tức đổi cả model.
- **deterministic true** để chạy lại cùng lệnh ra cùng kết quả.

Theo dõi dòng `=== end of epoch N | best AUC so far: ... ===`.

> ⚠️ **12 giờ là hạn cứng của Kaggle.** Nếu sau ~1 giờ mà chưa xong 1 epoch thì 10 epoch sẽ không
> kịp — dừng lại và giảm `BASELINE_EPOCHS`, hoặc chạy bằng Save & Run All (Commit).

> Đây là **điểm dữ liệu thứ ba**, không phải nỗ lực đạt tới một con số. Bạn đã có 87.55 và 88.13.
> Ba điểm là tối thiểu để viết `trung bình ± độ lệch chuẩn`.
> Đừng chạy đi chạy lại rồi chọn lần đẹp nhất — đó là chọn kết quả theo tập test.

In [ ]:
import time

trained_tags = []
for seed in SEEDS:
    print('#' * 78)
    print('TRAIN seed', seed)
    started = time.time()
    tag = train_baseline(seed)
    print(f'Train xong sau {(time.time() - started) / 3600:.2f} giờ')
    trained_tags.append(tag)
    score_model(OUT_ROOT / tag / f'model_{tag}.pth', tag)
print()
print('Đã train:', trained_tags)

## 7. Đường Cong Theo Epoch

`ucf_train.py` chấm toàn tập test ~12 lần mỗi epoch, nên log đã chứa sẵn đường cong.

`AUC1` cao nhất phải bằng con số §6 vừa in ra — vì file trọng số cuối chính là checkpoint tốt nhất đó.

> `ucf_train.py` chọn checkpoint theo AUC **trên chính tập test**. Đây là chọn-model-trên-test
> nên con số lạc quan hơn thực tế. Giữ nguyên vì đó là hành vi upstream, nhưng phải ghi rõ khi báo cáo.

In [ ]:
import pandas as pd

TRAIN_CURVE_RE = re.compile(
    r'epoch:\s+(\d+)\s+\|\s+step:\s+(\d+).*?'
    r'AUC1:\s+([\d.]+)\s+AP1:\s+([\d.]+).*?'
    r'AUC2:\s+([\d.]+)\s+AP2:\s+([\d.]+).*?'
    r'average MAP:\s+([\d.]+)', re.S)


def training_curve(tag):
    text = (LOG_DIR / f'train_{tag}.log').read_text(encoding='utf-8')
    return pd.DataFrame([
        dict(epoch=int(a), step=int(b), AUC1=float(c) * 100, AP1=float(d) * 100,
             AUC2=float(e) * 100, AP2=float(f) * 100, avgMAP=float(g))
        for a, b, c, d, e, f, g in TRAIN_CURVE_RE.findall(text)])


for tag in (trained_tags or [f'baseline_seed{s}' for s in SEEDS]):
    if not (LOG_DIR / f'train_{tag}.log').exists():
        print('Chưa có log cho', tag)
        continue
    curve = training_curve(tag)
    if curve.empty:
        print(tag, '- log chưa có điểm đánh giá nào.')
        continue
    print('=' * 78)
    print(tag, '-', len(curve), 'lần đánh giá')
    print(curve.groupby('epoch')[['AUC1', 'AUC2', 'avgMAP']].max().round(2).to_string())
    best = curve.loc[curve['AUC1'].idxmax()]
    print(f'  Tốt nhất: epoch {int(best.epoch)}, step {int(best.step)} -> '
          f'AUC1={best.AUC1:.2f} AP1={best.AP1:.2f} AUC2={best.AUC2:.2f} avgMAP={best.avgMAP:.2f}')
    curve.to_csv(RESULT_DIR / f'curve_{tag}.csv', index=False)

    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(curve.index, curve['AUC1'], label='AUC1 (phân loại)')
        ax.plot(curve.index, curve['AUC2'], label='AUC2 (đối chiếu)')
        ax.axhline(PAPER_METRICS['AUC1'], ls='--', c='gray', lw=1, label='paper 88.02 (tham chiếu)')
        ax.set_xlabel('lần đánh giá'); ax.set_ylabel('AUC (%)')
        ax.set_title(tag); ax.legend(); ax.grid(alpha=0.3)
        fig.tight_layout(); fig.savefig(RESULT_DIR / f'curve_{tag}.png', dpi=120); plt.show()
    except Exception as exc:
        print('  Bỏ qua phần vẽ:', exc)

## 8. Gom Sản Phẩm

Trên Kaggle, mọi thứ trong `/kaggle/working` tự thành Output của notebook — không cần copy đi đâu.
Cell này chỉ liệt kê và cảnh báo nếu vượt hạn mức 20 GB.

Tải về bằng tab **Output** ở panel bên phải, hoặc **Save Version** để giữ vĩnh viễn.

In [ ]:
total = 0
print('Nội dung /kaggle/working:')
for f in sorted(WORK.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        total += size
        if size > 1e6:
            print(f'  {size / 1e6:8.0f} MB  {f.relative_to(WORK)}')
print()
print(f'Tổng: {total / 1e9:.2f} GB / 20 GB')
if total > 18e9:
    print('  GẦN CHẠM HẠN MỨC. Xoá bớt epoch_checkpoints hoặc checkpoint.pth.')

print()
print('File nhỏ (log, csv, png):')
for f in sorted(RESULT_DIR.rglob('*')):
    if f.is_file() and f.stat().st_size <= 1e6:
        print(f'  {f.stat().st_size / 1e3:8.0f} KB  {f.relative_to(WORK)}')

## 9. Bảng Tổng Hợp

Gom mọi `score_*.log` thành một bảng, cộng thêm thống kê cho các lần baseline.

Cách đọc, và cũng là cách nên viết vào báo cáo:

- **`paper`** là dòng tham chiếu, chứng tỏ bản cài đặt trung thực. **Không** dùng làm cột hơn thua.
- **Các dòng `baseline_*`** là mốc so sánh thật. Độ lệch chuẩn của chúng là mức nhiễu.
- **`v0`** chỉ hơn/kém có ý nghĩa khi chênh lệch so với `baseline_ctrl` **lớn hơn** mức nhiễu đó.

In [ ]:
rows = []
for log in sorted(LOG_DIR.glob('score_*.log')):
    name = log.stem.replace('score_', '')
    scores = read_score(name)
    if scores:
        rows.append({'model': name, **scores})

if not rows:
    print('Chưa có kết quả nào. Chạy §5 trước.')
else:
    table = pd.DataFrame(rows).set_index('model').round(2)
    display(table)

    base = table[table.index.str.startswith('baseline')]
    if len(base) >= 2:
        print()
        print(f'Thống kê trên {len(base)} lần baseline ({", ".join(base.index)}):')
        for col in ['AUC1', 'avgMAP']:
            print(f'  {col:7s} trung bình {base[col].mean():6.2f}  '
                  f'độ lệch chuẩn {base[col].std():5.2f}  '
                  f'khoảng [{base[col].min():.2f}, {base[col].max():.2f}]')
        print()
        if 'v0' in table.index and 'baseline_ctrl' in table.index:
            print('v0 so với baseline_ctrl (cặp cùng script, cùng seed, khác đúng lambda):')
            for col in ['AUC1', 'AP1', 'AUC2', 'AP2', 'avgMAP']:
                delta = table.loc['v0', col] - table.loc['baseline_ctrl', col]
                noise = base[col].std() if col in base else float('nan')
                verdict = 'trong nhiễu' if abs(delta) < noise else 'VƯỢT nhiễu'
                print(f'  {col:7s} {delta:+6.2f}   (độ lệch chuẩn baseline {noise:.2f})  -> {verdict}')
    elif len(base) == 1:
        print()
        print('Mới có 1 lần baseline. Cần ít nhất 3 để tính độ lệch chuẩn — thêm seed vào SEEDS ở §1.')

    table.to_csv(RESULT_DIR / 'baseline_summary.csv')
    print()
    print('Saved:', RESULT_DIR / 'baseline_summary.csv')

## Ghi Chú

**Khác biệt so với bản Colab.** Chỉ §1 (dò dataset thay vì mount Drive) và §3 (nạp sẵn trọng số
CLIP thay vì copy feature — trên Kaggle `/kaggle/input` đã là đĩa local nên đọc thẳng). Từ §4 trở
đi mã nguồn giống hệt.

**Con số nào so với con số nào.** `v0` chỉ so được với `baseline_ctrl` — hai lần chạy đó cùng
script, cùng seed, cùng thứ tự dữ liệu, khác đúng một biến là λ. Các lần `baseline_seed*` phục vụ
việc khác: đo xem con số nhảy bao nhiêu khi **không đổi gì cả**.

**Đừng chọn lần chạy đẹp nhất.** Nếu chạy lại tới khi ra số ưng ý rồi lấy nó làm baseline, đó là
chọn kết quả theo tập test ở cấp độ lần-chạy.

**Kết luận "chưa chứng minh được" vẫn là kết luận hợp lệ** và đáng viết vào báo cáo.

**Chi tiết code:** `baseline/README.md` liệt kê đủ 4 file đã sửa so với upstream và lý do từng chỗ.